In [1]:

import sys, os
sys.path.append(os.path.abspath("..")) 
import json
import torch
from datasets import CNFDataset
from models import LightningModelCNF
from utils import args_cnf, gen_image_cnf, plotly_generate

**Read configuration files and arguments:**

In [2]:
# Arguments
parser = args_cnf()
args, unknown = parser.parse_known_args()

args.particle = "proton_contained"

args.metadata_path = "/scratch2/libota/SFGD_Vertex_Activity/Data/GenValid/metadata.pkl"
args.dataset_path = "/scratch2/libota/SFGD_Vertex_Activity/Data/GenValid/{}/{}/{}/{}.zip"
args.cnf_ind_path = "/scratch2/libota/SFGD_Vertex_Activity/Data/GenValid/gan_ind.pkl"
args.checkpoint_path = "/scratch2/libota/SFGD_Vertex_Activity/Results/cnf/test_spline/checkpoints"
args.checkpoint_name = args.particle

if args.particle == "muon" or args.particle == "proton_exiting":
    args.label_size = 10
elif args.particle == "proton_contained":
    args.label_size = 7
args.epochs = 50
args.log_every_n_steps = 2000
args.batch_size = 512
args.hidden = 256
args.num_workers = 64

valid_params_folder_base = "/scratch2/libota/SFGD_Vertex_Activity/Data/GenValid/proton_contained/analysis_files"

**Load the pre-trained weights of the different generative-adversarial-network (GAN) models:**

In [3]:

checkpoint_path = "/".join((args.checkpoint_path, args.particle, "train_loss","last.ckpt"))
# Load weights of pre-trained generator models
#checkpoint_p = torch.load(checkpoint_path, map_location='cpu')

# state_dict = {
#     key.replace("generator.", ""): value for key, value in checkpoint_p['state_dict'].items()
# }

# generator.load_state_dict(state_dict, strict=False)
# generator.eval();
#
# set the parameters of the model

model = LightningModelCNF.load_from_checkpoint(checkpoint_path, img_shape = (args.img_size, args.img_size, args.img_size), 
                                    label_size = args.label_size, 
                                    hidden_features = args.hidden, 
                                    num_blocks_in_MADE = args.num_blocks_in_MADE, 
                                    num_transformers = args.num_transformers, 
                                    lr = args.lr, 
                                    wd = args.weight_decay)

model.eval()

# move to cpu
model.to("cpu")


LightningModelCNF(
  (nflow): Flow(
    (_transform): CompositeTransform(
      (_transforms): ModuleList(
        (0): MaskedPiecewiseRationalQuadraticAutoregressiveTransform(
          (autoregressive_net): MADE(
            (initial_layer): MaskedLinear(in_features=125, out_features=256, bias=True)
            (context_layer): Linear(in_features=7, out_features=256, bias=True)
            (activation): ReLU()
            (blocks): ModuleList(
              (0-1): 2 x MaskedResidualBlock(
                (context_layer): Linear(in_features=7, out_features=256, bias=True)
                (linear_layers): ModuleList(
                  (0-1): 2 x MaskedLinear(in_features=256, out_features=256, bias=True)
                )
                (activation): ReLU()
                (dropout): Dropout(p=0.0, inplace=False)
              )
            )
            (final_layer): MaskedLinear(in_features=256, out_features=3625, bias=True)
          )
        )
        (1): BatchNorm()
        (2)

In [ ]:
print(checkpoint_p['state_dict'].keys())

In [4]:
print(test_set_p.metadata['statistics']['per_tree']['proton_contained']['recon_charge']['max'])
print(test_set_p.metadata['statistics']['per_tree']['proton_contained']['recon_charge']['min'])

2658
3


In [4]:
for key, value in checkpoint_p['state_dict'].items():
    print(key, value.shape)

generator.bert.cls_token torch.Size([1, 1, 64])
generator.bert.embedding.input.weight torch.Size([64, 1])
generator.bert.embedding.input.bias torch.Size([64])
generator.bert.embedding.label.embedding.weight torch.Size([64, 10])
generator.bert.embedding.label.embedding.bias torch.Size([64])
generator.bert.embedding.position.vol_idx torch.Size([126])
generator.bert.embedding.position.embedding.weight torch.Size([126, 64])
generator.bert.embedding.noise.weight torch.Size([64, 512])
generator.bert.embedding.noise.bias torch.Size([64])
generator.bert.transformer_blocks.0.attention.linear_layers.0.weight torch.Size([64, 64])
generator.bert.transformer_blocks.0.attention.linear_layers.0.bias torch.Size([64])
generator.bert.transformer_blocks.0.attention.linear_layers.1.weight torch.Size([64, 64])
generator.bert.transformer_blocks.0.attention.linear_layers.1.bias torch.Size([64])
generator.bert.transformer_blocks.0.attention.linear_layers.2.weight torch.Size([64, 64])
generator.bert.transforme

In [9]:
print(checkpoint_p['state_dict']['critic.bert.embedding.input.weight'])

tensor([[ 0.5004],
        [-2.5577],
        [ 0.5532],
        [-9.0691],
        [ 2.0405],
        [ 1.9576],
        [ 0.0836],
        [ 0.5649],
        [-0.5833],
        [-0.5987],
        [ 1.0329],
        [ 5.5730],
        [ 1.9099],
        [ 4.0439],
        [ 0.7667],
        [ 1.3041],
        [ 0.2591],
        [ 1.5450],
        [ 6.0901],
        [ 2.3652],
        [ 3.0429],
        [-3.6318],
        [ 2.1024],
        [-0.3306],
        [ 0.4442],
        [ 5.3540],
        [ 5.7043],
        [-0.4855],
        [ 2.4442],
        [-0.2827],
        [-0.6130],
        [ 0.6491],
        [-1.9083],
        [ 1.0670],
        [-0.6086],
        [ 0.6371],
        [ 1.2413],
        [-1.5205],
        [ 5.3508],
        [ 0.5478],
        [-3.5981],
        [-1.0268],
        [ 0.4822],
        [ 0.7821],
        [-2.2264],
        [ 1.6432],
        [ 1.1424],
        [ 1.7548],
        [ 0.8087],
        [ 2.2181],
        [ 0.6237],
        [ 0.8127],
        [ 1.

In [9]:
import pandas as pd
import pickle as pkl
import sys
sys.path.append('/opt/software/root/6.36.00/lib') 
#!pip install uproot
from ROOT import TFile, TTree, std
import ROOT
import matplotlib.pyplot as plt
import glob
params_file = valid_params_folder_base + "/proton_0/proton_param.csv"

metadata = pkl.load(open(args.metadata_path, "rb"))

param = pd.read_csv(params_file)

a=param.iloc[:].values.reshape(-1)
print(a)

[   0.59    0.21 -192.05    0.92   -0.3     0.26    5.21]


**Run each GAN on some arbitrary input kinematics:**

In [7]:
'''
Proton GAN
'''
import numpy as np
import pandas as pd
# Set your kinematics here:
# ke = 30.3  # Initial kinetic energy
# ini_dir = [0.9999999999999999, 0.0, 0.0]  # Initial direction
# ini_pos = [-1.5, -4.2, 2.7]  # Initial 3D position (mm)

params_file = valid_params_folder_base + "/proton_0/proton_param.csv"

param = pd.read_csv(params_file).iloc[:].values.reshape(-1)

source_center_vec = [0.01,0.02,-192.87];
# convert torch tensor to numpy array
ke = float(param[6]) - metadata['statistics']['per_tree'][args.particle]['true_iniekin']['mean']
ke /= metadata['statistics']['per_tree'][args.particle]['true_iniekin']['std']
ini_pos = (param[0:3] - source_center_vec - args.cube_size/20) * 10/(args.cube_size * 1.5)
ini_dir = param[3:6]

if args.particle == "proton_exiting" or args.particle == "muon":
    exit_pos = param[7:10]
else:
    exit_pos = None

#print(ke, ini_pos, ini_dir, exit_pos, img)

if exit_pos is not None:
    params = np.array([ini_pos[0], ini_pos[1], ini_pos[2], ke, ini_dir[0], ini_dir[1], ini_dir[2], exit_pos[0], exit_pos[1], exit_pos[2]])
else:
    params = np.array([ini_pos[0], ini_pos[1], ini_pos[2], ke, ini_dir[0], ini_dir[1], ini_dir[2]])

labels = torch.tensor([params], dtype=torch.float32)

print(labels)




tensor([[ 0.0432, -0.2100,  0.1990, -1.5169,  0.9200, -0.3000,  0.2600]])


/tmp/ipykernel_2945458/2596128563.py:34: UserWarning: Creating a tensor from a list of numpy.ndarrays is extremely slow. Please consider converting the list to a single numpy.ndarray with numpy.array() before converting to a tensor. (Triggered internally at /pytorch/torch/csrc/utils/tensor_new.cpp:253.)
  labels = torch.tensor([params], dtype=torch.float32)


In [ ]:
data_file = "/scratch2/libota/SFGD_Vertex_Activity/Data/GenValid/proton_contained/analysis_files/proton_0/output_PGun_proton_contained.root"
data_tree = TFile(data_file, "read")
ana_tree = data_tree["ana"]
n_entry = ana_tree.GetEntries()

# dictionary for 5 x 5 x 5 grid
hit_charge = {}
for i in range(5):
    for j in range(5):
        for k in range(5):
            hit_charge[(i, j, k)] = []  

print(hit_charge.keys())
print ("number of entries: ", n_entry)
for i in range(n_entry):
    ana_tree.GetEntry(i)
    x = ana_tree.recon_sfg_hitposx_rel
    y = ana_tree.recon_sfg_hitposy_rel
    z = ana_tree.recon_sfg_hitposz_rel
    for ind in range(len(x)):
        pos = np.array([x[ind]+2, y[ind]+2, z[ind]+2])
        
        hit_charge[tuple(pos)].append(ana_tree.recon_sfg_charge[ind])

# for key, value in hit_charge.items():
#     print(key, len(value))


        



dict_keys([(0, 0, 0), (0, 0, 1), (0, 0, 2), (0, 0, 3), (0, 0, 4), (0, 1, 0), (0, 1, 1), (0, 1, 2), (0, 1, 3), (0, 1, 4), (0, 2, 0), (0, 2, 1), (0, 2, 2), (0, 2, 3), (0, 2, 4), (0, 3, 0), (0, 3, 1), (0, 3, 2), (0, 3, 3), (0, 3, 4), (0, 4, 0), (0, 4, 1), (0, 4, 2), (0, 4, 3), (0, 4, 4), (1, 0, 0), (1, 0, 1), (1, 0, 2), (1, 0, 3), (1, 0, 4), (1, 1, 0), (1, 1, 1), (1, 1, 2), (1, 1, 3), (1, 1, 4), (1, 2, 0), (1, 2, 1), (1, 2, 2), (1, 2, 3), (1, 2, 4), (1, 3, 0), (1, 3, 1), (1, 3, 2), (1, 3, 3), (1, 3, 4), (1, 4, 0), (1, 4, 1), (1, 4, 2), (1, 4, 3), (1, 4, 4), (2, 0, 0), (2, 0, 1), (2, 0, 2), (2, 0, 3), (2, 0, 4), (2, 1, 0), (2, 1, 1), (2, 1, 2), (2, 1, 3), (2, 1, 4), (2, 2, 0), (2, 2, 1), (2, 2, 2), (2, 2, 3), (2, 2, 4), (2, 3, 0), (2, 3, 1), (2, 3, 2), (2, 3, 3), (2, 3, 4), (2, 4, 0), (2, 4, 1), (2, 4, 2), (2, 4, 3), (2, 4, 4), (3, 0, 0), (3, 0, 1), (3, 0, 2), (3, 0, 3), (3, 0, 4), (3, 1, 0), (3, 1, 1), (3, 1, 2), (3, 1, 3), (3, 1, 4), (3, 2, 0), (3, 2, 1), (3, 2, 2), (3, 2, 3), (3, 2, 4),

Warning in <TClass::Init>: no dictionary for class PackageVersion is available
Warning in <TClass::Init>: no dictionary for class TrackCategoryDefinition is available
Warning in <TClass::Init>: no dictionary for class TrackTypeDefinition is available
Warning in <TClass::Init>: no dictionary for class SelectionBase is available
Warning in <TClass::Init>: no dictionary for class StepBase is available
Warning in <TClass::Init>: no dictionary for class DocString is available
Warning in <TClass::Init>: no dictionary for class ConfigurationBase is available
Warning in <TClass::Init>: no dictionary for class ToyVariationWrite is available
Warning in <TClass::Init>: no dictionary for class SystematicBase is available
Warning in <TClass::Init>: no dictionary for class CorrectionBase is available
Warning in <TClass::Init>: no dictionary for class Header is available


In [11]:
# check the max hit in each voxel
max_hit = 0
for key, value in hit_charge.items():
    if len(value) > 0:
        print(key, max(value))


(1, 2, 2) 26.131175994873047
(2, 1, 2) 32.02325439453125
(2, 2, 1) 34.785003662109375
(2, 2, 2) 244.65020751953125
(2, 2, 3) 31.63243865966797
(2, 3, 2) 29.660762786865234
(3, 2, 2) 28.0672607421875


In [14]:
print(labels.size(0))

1


In [ ]:
n_sample = 10

generated_charge = []

for n in range(100):

    print(n)
    with torch.no_grad():
        generated_p = model.sample(labels, num_samples=n_sample)
    print(generated_p.shape)
    generated_p_1 = generated_p[0]

    min_charge = 0
    max_charge = metadata['statistics']['per_tree'][args.particle]['recon_charge']['max']

    generated_p_1 = (generated_p_1 + 1) / 2
    generated_p_1 *= (max_charge - min_charge)
    generated_p_1 += min_charge

    for s in range(n_sample):
        generated_p_s = generated_p_1[s]
        generated_charge.append(generated_p_s.detach().cpu().numpy())
        
print(len(generated_charge))
samples = np.array(generated_charge)
samples = samples.reshape(samples.shape[0], 5, 5, 5)

print(samples.shape)

    # generated_p_1 = generated_p_1.reshape(5, 5, 5)

    # for i in range(5):
    #     for j in range(5):
    #         for k in range(5):
    #             generated_charge[(i, j, k)].append(generated_p_1[i, j, k])


0


KeyboardInterrupt: 

In [17]:
for key, value in generated_charge.items():
    if len(value) > 0:
        print(key, max(value))

(0, 0, 0) tensor(146.9201)
(0, 0, 1) tensor(9.3666)
(0, 0, 2) tensor(9.4907)
(0, 0, 3) tensor(12.0099)
(0, 0, 4) tensor(6.1992)
(0, 1, 0) tensor(5.0517)
(0, 1, 1) tensor(7.3287)
(0, 1, 2) tensor(7.2653)
(0, 1, 3) tensor(10.3159)
(0, 1, 4) tensor(10.5314)
(0, 2, 0) tensor(6.8478)
(0, 2, 1) tensor(7.4350)
(0, 2, 2) tensor(15.8352)
(0, 2, 3) tensor(9.6627)
(0, 2, 4) tensor(9.6647)
(0, 3, 0) tensor(11.8682)
(0, 3, 1) tensor(10.7325)
(0, 3, 2) tensor(13.7056)
(0, 3, 3) tensor(9.2287)
(0, 3, 4) tensor(9.7437)
(0, 4, 0) tensor(7.7817)
(0, 4, 1) tensor(14.4957)
(0, 4, 2) tensor(7.8003)
(0, 4, 3) tensor(8.5778)
(0, 4, 4) tensor(8.5789)
(1, 0, 0) tensor(4.6870)
(1, 0, 1) tensor(13.1506)
(1, 0, 2) tensor(11.3415)
(1, 0, 3) tensor(5.5003)
(1, 0, 4) tensor(10.6105)
(1, 1, 0) tensor(8.3499)
(1, 1, 1) tensor(5.1848)
(1, 1, 2) tensor(3.5860)
(1, 1, 3) tensor(12.3105)
(1, 1, 4) tensor(9.2037)
(1, 2, 0) tensor(7.5219)
(1, 2, 1) tensor(9.8251)
(1, 2, 2) tensor(6.6223)
(1, 2, 3) tensor(18.6285)
(1, 2, 4) 

In [18]:
# change the tensors to numpy arrays
#generated_charge = {key: np.array(value) for key, value in generated_charge.items()}
hit_charge = {key: np.array(value) for key, value in hit_charge.items()}





In [11]:
print(max(generated_charge[2,2,2]))

1742.877


In [22]:
import matplotlib.pyplot as plt
# plot the charge distribution in each voxel
# generated vs true
for key, value in hit_charge.items():
    if hit_charge[key].size > 0:
        plt.hist(hit_charge[key], bins=100, alpha=0.5, label="true", color="blue", density=True)
        generated_charge = samples[:,key[0],key[1],key[2]]
        generated_charge = generated_charge[generated_charge>5]
        print(generated_charge)
        plt.hist(generated_charge, bins=100, alpha=0.5, label="generated", color="red", density=True)
        plt.legend()
        plt.savefig(f"charge_distribution_{key}.png")
        plt.close() 






[26.40499  20.131353 18.370018]
[27.081245 28.682013 31.864616 26.778803]
[25.124249 18.064804 21.709387]
[214.94217  173.00906  153.35228  159.1736   172.15338  167.61472
 154.75969  140.6991   161.51242  176.71947  113.39859  196.83482
 190.66472  174.55518  147.44716  176.34526  167.49866  160.68059
 184.51538  163.15366  176.53998  160.40001  185.44077  213.40509
 158.69293  189.16052  176.12045  165.03073  135.69528  170.1533
 149.89163  165.39218  171.59651  161.50473  192.98912  134.04349
 149.69495  173.90245  138.86125  164.38948  144.7602   146.53809
 144.05447  149.90819  122.675255 135.37675  157.4689   177.82887
 135.27933  157.09502  138.21367  152.76134  193.00932  151.266
 154.41383  158.27333  190.20212  165.19272  154.99527  170.20804
 150.13135  188.40465  176.31493  148.41429  199.42166  185.77377
 180.26924  142.99149  170.19022  192.00836  131.34355  137.11235
 157.77611  174.18571  159.7045   174.40387  172.72208  188.37439
 168.67975  145.50972  165.75934  168.0

In [23]:
with torch.no_grad():
    generated_p = model.sample(labels)

print(generated_p)

tensor([[[-1.0127, -1.0007, -0.9983, -0.9970, -0.9995, -0.9985, -1.0009,
          -0.9981, -0.9962, -0.9976, -1.0064, -1.0001, -1.0002, -0.9993,
          -1.0002, -0.9997, -0.9956, -0.9971, -0.9986, -0.9968, -0.9990,
          -0.9968, -1.0020, -1.0001, -1.0054, -0.9995, -1.0018, -0.9990,
          -0.9989, -1.0042, -0.9997, -1.0004, -1.0037, -0.9992, -1.0006,
          -0.9985, -0.9984, -0.9991, -1.0002, -1.0032, -1.0006, -0.9988,
          -0.9994, -0.9933, -1.0016, -0.9980, -0.9983, -0.9981, -1.0019,
          -1.0043, -1.0001, -0.9984, -0.9990, -1.0023, -1.0006, -1.0008,
          -0.9987, -1.0016, -1.0004, -1.0028, -0.9980, -0.9995, -1.3637,
          -0.7763, -1.0009, -0.9965, -0.9966, -0.9986, -0.9793, -0.9983,
          -0.9992, -0.9977, -1.0001, -0.9999, -0.9996, -1.0026, -0.9987,
          -0.9983, -0.9995, -1.0017, -1.0006, -0.9964, -0.9952, -1.0009,
          -0.9991, -0.9980, -1.0012, -1.0033, -0.9927, -1.0015, -0.9927,
          -1.0038, -0.9984, -1.0012, -0.9999, -1.00

tensor([[[  3.0000,   3.0000,   3.0000,   3.0000,   3.0000],
         [  3.0000,   3.0000,   3.0000,   3.0000,   3.0000],
         [  3.0000,   3.0000,   3.0000,   3.0000,   3.0000],
         [  3.0000,   3.0000,   3.0000,   3.0000,   3.0000],
         [  3.0000,   3.0000,   3.0000,   3.0000,   3.0000]],

        [[  3.0000,   3.0000,   3.0000,  39.9746,   3.0000],
         [  3.0000,   3.0000, 230.2248,   3.0000,   3.0000],
         [  3.0000,   3.0000, 100.7135,   3.0000,   3.0000],
         [  3.0000,   3.0000,   3.0000,   3.0000,   3.0000],
         [  3.0000,   3.0000,   3.0000,   3.0000,   3.0000]],

        [[  3.0000,   3.0000,   3.0000,   3.0000,   3.0000],
         [  3.0000,   3.0000, 228.2390,   3.0000,   3.0000],
         [  3.0000,   3.0000, 187.8737,   3.3160,   3.0000],
         [  3.0000,   3.0000,   3.0000,   3.0000,   3.0000],
         [  3.0000,   3.0000,   3.0000,   3.0000,   3.0000]],

        [[  3.0000,   3.0000,   3.0000,   3.0000,   3.0000],
         [  3.0000

**Visualise the GAN-generated images:**

In [24]:
'''
Plot the generated images!
'''


#generated_p = generated_p.numpy()
# copy the image to a pure numpy array

print(generated_p.shape)
generated_p_1 = generated_p[0][0]

min_charge = 0
max_charge = metadata['statistics']['per_tree'][args.particle]['recon_charge']['max']

generated_p_1 = (generated_p_1 + 1) / 2
generated_p_1 *= (max_charge - min_charge)
generated_p_1 += min_charge

print(generated_p_1.shape)
# reshape the generated image to a 5x5x5 array
generated_p_1 = generated_p_1.reshape(5, 5, 5)



generated_plot = np.zeros((5, 5, 5))

for i in range(generated_plot.shape[0]):
    for j in range(generated_plot.shape[1]):
        for k in range(generated_plot.shape[2]):
            generated_plot[i, j, k] = float(generated_p_1[i, j, k])
print(generated_plot)
# check the type of the elements in the array
print(generated_plot.dtype)

# Max deposited energy in one voxel
max_energy = generated_plot.max()

#generated_plot[generated_plot > 150] = 0
generated_plot[generated_plot < 0] = 0
print(max_energy)




torch.Size([1, 1, 125])
torch.Size([125])
[[[-1.68239079e+01 -9.56278324e-01  2.25761533e+00  4.03875494e+00
    6.23101830e-01]
  [ 2.00444555e+00 -1.17459369e+00  2.48654556e+00  5.01467848e+00
    3.15361142e+00]
  [-8.51350689e+00 -1.49240255e-01 -2.29405403e-01  9.24275637e-01
   -3.18284154e-01]
  [ 3.58683586e-01  5.88302851e+00  3.83232188e+00  1.81646943e+00
    4.26103115e+00]
  [ 1.37167954e+00  4.28432035e+00 -2.68204689e+00 -1.72054052e-01
   -7.16479969e+00]]

 [[ 7.28061140e-01 -2.40986586e+00  1.33706284e+00  1.52123666e+00
   -5.57163620e+00]
  [ 4.32432353e-01 -5.21073461e-01 -4.96105003e+00  1.04151320e+00
   -7.50320435e-01]
  [ 2.05118227e+00  2.07138181e+00  1.18497086e+00 -2.84855604e-01
   -4.29992533e+00]
  [-8.37456465e-01  1.55735850e+00  7.53092945e-01  8.85159492e+00
   -2.08286810e+00]
  [ 2.63459754e+00  2.30023289e+00  2.46381092e+00 -2.47735643e+00
   -5.74020481e+00]]

 [[-1.60964012e-01  2.17642045e+00  1.29777241e+00 -3.07827830e+00
   -7.39547253e-0

In [14]:
generated_plot = generated_plot.astype(np.float32)

In [25]:

print("- Proton image:")
plotly_generate(generated_plot, max_energy=max_energy)


- Proton image:


In [ ]:
event = test_set_p[15]
true_img = np.zeros((125,))

for i in range(true_img.shape[0]):
    true_img[i] = float(event["image"][i])

true_img = true_img.reshape(5, 5, 5)

# get back the normalization
min_charge = 0
max_charge = test_set_p.metadata['statistics']['per_tree'][args.particle]['recon_charge']['max']
true_img = (true_img + 1) / 2
true_img *= (max_charge - min_charge)
true_img += min_charge

print(test_set_p.metadata['statistics']['per_tree']['proton_contained']['recon_charge']['std'])
print(test_set_p.metadata['statistics']['per_tree']['proton_contained']['recon_charge']['mean'])

max_energy = true_img.max()

print(true_img)

print("- True image:")
plotly_generate(true_img, max_energy=max_energy)



303.3067761656923
197.1132585084357
[[[-8.43769499e-14 -8.43769499e-14 -8.43769499e-14 -8.43769499e-14
   -8.43769499e-14]
  [-8.43769499e-14 -8.43769499e-14 -8.43769499e-14 -8.43769499e-14
   -8.43769499e-14]
  [-8.43769499e-14 -8.43769499e-14 -8.43769499e-14 -8.43769499e-14
   -8.43769499e-14]
  [-8.43769499e-14 -8.43769499e-14 -8.43769499e-14 -8.43769499e-14
   -8.43769499e-14]
  [-8.43769499e-14 -8.43769499e-14 -8.43769499e-14 -8.43769499e-14
   -8.43769499e-14]]

 [[-8.43769499e-14 -8.43769499e-14 -8.43769499e-14  2.35980667e+02
   -8.43769499e-14]
  [-8.43769499e-14 -8.43769499e-14  3.43481979e+01  3.48438988e+01
   -8.43769499e-14]
  [-8.43769499e-14 -8.43769499e-14 -8.43769499e-14 -8.43769499e-14
   -8.43769499e-14]
  [-8.43769499e-14 -8.43769499e-14 -8.43769499e-14 -8.43769499e-14
   -8.43769499e-14]
  [-8.43769499e-14 -8.43769499e-14 -8.43769499e-14 -8.43769499e-14
   -8.43769499e-14]]

 [[-8.43769499e-14 -8.43769499e-14 -8.43769499e-14 -8.43769499e-14
   -8.43769499e-14]
  [

In [16]:
print(event["image"].numpy())

[ 197.11325851  197.11325851  197.11325851  197.11325851  197.11325851
  197.11325851  197.11325851  197.11325851  197.11325851  197.11325851
  197.11325851  197.11325851  197.11325851  197.11325851  197.11325851
  197.11325851  197.11325851  197.11325851  197.11325851  197.11325851
  197.11325851  197.11325851  197.11325851  197.11325851  197.11325851
  197.11325851  197.11325851  197.11325851  197.11325851  197.11325851
  197.11325851  197.11325851  197.11325851  197.11325851  197.11325851
  197.11325851  197.11325851  266.32901016  197.11325851  197.11325851
  197.11325851  197.11325851  197.11325851  197.11325851  197.11325851
  197.11325851  197.11325851  197.11325851  197.11325851  197.11325851
  197.11325851  197.11325851  197.11325851  197.11325851  197.11325851
  197.11325851  197.11325851  261.34530654  197.11325851  197.11325851
  197.11325851  263.76361099 1348.25107589  298.85057845  197.11325851
  197.11325851  197.11325851  286.39696517  197.11325851  197.11325851
  197.